# Finding and diagnosing bugs with a property test: IM2 hillslope routing

ELM's IM2 hillslope hydrology (`use_IM2_hillslope_hydrology`) routes part of each column's surface
runoff to the next lower topounit of its gridcell: the water waits in that topounit's store
(`top_ws%from_uphill`), half of the store is released into its soil every step, and the rest of
ELM's step drains the store by what was released.

This notebook starts from a property test of that routing, written without knowing of any bug, and
follows what its failures lead to:

1. the property test runs on random hillslopes and fails;
2. the shrunk failures, read side by side, suggest hypotheses;
3. a targeted spec confirms each hypothesis on declared hillslopes, and measures what it does;
4. the confirmed hypotheses are traced to ELM's code;
5. a run with the suspected cause removed proves the diagnosis.

The specs are built in `specs.py` (each named after what it tests) and written as TOML next to this
notebook, so `spel proptest run <name>.proptest.toml` runs the same checks; the analyses are the
named functions of `analysis.py`. `README.md` is the report, with what the bugs do in real ELM runs.

In [1]:
import io
import time
from pathlib import Path

import pandas as pd

from spel.proptest import api as pt, runner
from spel.proptest.experiments.hillslope_im2 import analysis, specs

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", None)
HERE = Path.cwd()  # spel/proptest/experiments/hillslope_im2
DTIME = 1800.0     # the case's time step (s)

## 0. The process under test

The case `hs_season` runs `HydrologyNoDrainage` (whose `SurfaceRunoff` releases the store) and
`HydrologyDrainage` (which sends runoff downhill) in elm_drv's order. Its reference is a capture of a
real IM2 run; it supplies constants, parameters and the statistics the input rules start from.

Over several steps, the case is not alone: between two calls ELM runs the rest of its step.
`coupling.boundary` finds every variable the case reads that something else in the step writes
(from elm_drv's static trace, under these namelist options):

In [2]:
case = pt.Case(specs.CASE, reference=specs.CASE)
boundary = analysis.boundary_table(case, specs.IM2_ON)
print(f"{len(boundary)} variables the routines read are written elsewhere in the step")
boundary.loc[["top_ws%from_uphill", "col_wf%qflx_rain_grnd", "col_ws%h2osoi_liq", "col_es%t_soisno",
              "col_wf%qflx_evap_tot", "col_pp%dz"]]

61 variables the routines read are written elsewhere in the step


,kind,writers,where
variable,,,
top_ws%from_uphill,inout,2,"colwaterbalancecheck (BalanceCheckMod.F90:322), after; and 1 more"
col_wf%qflx_rain_grnd,input,1,"p2c_1d_filter (subgridAveMod.F90:333) in elm_drv_patch2col, before"
col_ws%h2osoi_liq,inout,6,"copy_state (dynInitColumnsMod.F90:284) in dynsubgrid_driver, before, conditional; and 5 more"
col_es%t_soisno,inout,21,"copy_state (dynInitColumnsMod.F90:276) in dynsubgrid_driver, before, conditional; and 20 more"
col_wf%qflx_evap_tot,input,1,"p2c_1d_filter (subgridAveMod.F90:333) in elm_drv_patch2col, before"
col_pp%dz,inout,3,"canopyhydrology (CanopyHydrologyMod.F90:670), before, conditional; and 2 more"


A multi-step spec must say what stands in for each of them, and the run refuses to start
otherwise:

* `forcing`: set before every step; here the rain on the ground, drawn for the first half of a
  storm, then 0;
* `between`: assigned after every step; here the store's drain (`specs.STORE_DRAIN`,
  ColWaterBalanceCheck's three lines as an expression);
* `keep`: left to the process; everything else. The inputs make that a statement about an isolated
  process: no snow, ice, evaporation or transpiration, warm soil, ELM's soil grid
  (`specs.isolated_process`);
* `carry`: outputs not reset between steps (`qflx_from_uphill` keeps ELM's cold-start 0 on columns
  the routines skip).

## 1. The property test fails

`specs.random_hillslopes` states what IM2 routing must do on *any* hillslope, and runs it over a
12-step storm on four drawn domains: gridcells of 1-3 topounits of soil, wetland, lake and plain
glacier, at random elevations and weights, some of weight 0. Nothing in it knows of a bug:

In [3]:
spec = specs.random_hillslopes(HERE / "random_hillslopes.proptest.toml")
spec.to_toml(spec.path)
parsed = spec.validate()
display(analysis.boundary_declarations(parsed, boundary.index))
analysis.check_table(parsed)

,steps,forcing,between,keep,UNDECLARED
scenario,,,,,
storm,12,1,1,59,0


,kind,scenarios,expression
check,,,
the flux from uphill is finite on every hydrology column,property,*,finite(col_wf%qflx_from_uphill(c))
the store stays finite and non-negative,property,*,finite(top_ws%from_uphill(t)) .and. top_ws%from_uphill(t) >= 0.0
no topounit sends water to itself,property,*,"implies(col_wf%qflx_to_downhill(c) > 0.0, top_pp%downhill_ti(t) /= t)"
water is conserved: soil + stores + runoff so far are the same with IM2 off,relation,*,"close(child_wsum(base%col_ws%endwb, 'g', filter%hydrologyc) + child_wsum(base%top_ws%from_uphill, 'g') + dtime_mod * acc(child_wsum(base%col_wf%qflx_runoff, 'g', filter%hydrologyc)), child_wsum(follow%col_ws%endwb, 'g', filter%hydrologyc) + dtime_mod * acc(child_wsum(follow%col_wf%qflx_runoff, 'g', filter%hydrologyc)), rtol=1.0e-9, atol=1.0e-9)"


In [4]:
t0 = time.time()
outcomes = runner.run(parsed, case.inv, settings={"seed": 0}, out=io.StringIO())
print(f"{time.time() - t0:.0f} s")
analysis.outcome_table(outcomes)

314 s


where,domain 1,domain 2,domain 3,domain 4
check,,,,
no topounit sends water to itself,pass,FAIL,pass,FAIL
the flux from uphill is finite on every hydrology column,FAIL,FAIL,pass,FAIL
the store stays finite and non-negative,FAIL,FAIL,pass,FAIL
water is conserved: soil + stores + runoff so far are the same with IM2 off,FAIL,FAIL,pass,FAIL


Each drawn domain is a new session; every failure is then shrunk to a smaller domain on which the
check still fails *the same way*: its failure signature (the routine aborted, or which terms of the
check were non-finite at the first failing element) must match, so the shrinking cannot slip from
one bug to another that trips the same check. Shrinking is most of the run's time.

## 2. Reading the shrunk failures

For each failure: the domain it came from, its signature, and the smallest domain it still fails on,
in that way (`soil(0)`: a landunit of weight 0; `[w0]`: a topounit of weight 0; `|` separates
topounits):

In [5]:
analysis.shrunk_domains(outcomes)

signature                                            shrunk_to
domain check                                                                                                                                                                              
1      the flux from uphill is finite on every hydrology column                                   non-finite: col_wf%qflx_from_uphill                                  d0.g3: soil(0)+lake
       the store stays finite and non-negative                                                         non-finite: top_ws%from_uphill                                  d0.g3: soil(0)+lake
       water is conserved: soil + stores + runoff so far are the same with IM2 off  non-finite: child_wsum(base%top_ws%from_uphill,g)                                  d0.g3: soil(0)+lake
2      the flux from uphill is finite on every hydrology column                                   non-finite: col_wf%qflx_from_uphill                               d1.g2: soil(0)+wetland
       the store stays finite and non-negative                                                         non-finite: top_ws%from_uphill                               d1.g2: soil(0)+wetland
       no topounit sends water to itself                                                                                finite values                                   d1.g2: soil | soil
       water is conserved: soil + stores + runoff so far are the same with IM2 off  non-finite: child_wsum(base%top_ws%from_uphill,g)                 d1.g2: soil | soil | soil(0)+wetland
4      the flux from uphill is finite on every hydrology column                                   non-finite: col_wf%qflx_from_uphill  d3.g1: soil [w0] | soil; d3.g2: soil | soil(0)+lake
       the store stays finite and non-negative                                                         non-finite: top_ws%from_uphill  d3.g1: soil [w0] | soil; d3.g2: soil | soil(0)+lake
       no topounit sends water to itself                                                                                finite values  d3.g1: soil [w0] | soil; d3.g2: soil | soil(0)+lake
       water is conserved: soil + stores + runoff so far are the same with IM2 off                                      finite values  d3.g1: soil [w0] | soil; d3.g2: soil | soil(0)+lake

In [6]:
failed = [o for o in outcomes if not o.passed]
print(analysis.failure_summaries(failed[:1] + [o for o in failed if o.check == specs.NO_SELF_SEND[0]][:1], lines=7))

the flux from uphill is finite on every hydrology column: finite(col_wf%qflx_from_uphill(c))
  after step 1 of 12
  fails at 2 of 2 elements; first: element 1 (set 0, d0.g3, soil, column soil)
    finite = False
    col_wf%qflx_from_uphill = nan
  drawn inputs at that element:
    aerosol_vars%mss_bcphi_col = [0. 0. 0. 0. 0.]

no topounit sends water to itself: implies(col_wf%qflx_to_downhill(c) > 0.0, top_pp%downhill_ti(t) /= t)
  after step 1 of 12
  fails at 2 of 4 elements; first: element 2 (set 0, d1.g2, topounit 2/2, soil, column soil)
    implies = False
    col_wf%qflx_to_downhill = 8.723098524619996e-05
    top_pp%downhill_ti = 2
  drawn inputs at that element:


Three different signatures; each shrunk domain is worth a look. The topounits of each, with the
downhill neighbour the domain gives the routines (`downhill_ti`, ELM's rule) and what is wrong with it:

In [7]:
self_send = next(o for o in failed if o.check == specs.NO_SELF_SEND[0])
analysis.topounit_table(analysis.shrunk_domain_layout(self_send))

,name,topo_number,weight,active,elevation,landunits,downhill_ti,points at,uphill_count,problem
topounit,,,,,,,,,,
1,d1.g2,1,0.5,True,177.5,soil,2,2,0,
2,d1.g2,2,0.5,True,102.6,soil,2,2,1,-> itself


In [8]:
nan_flux = next(o for o in failed if o.check == specs.FLUX_FINITE[0])
analysis.topounit_table(analysis.shrunk_domain_layout(nan_flux))

,name,topo_number,weight,active,elevation,landunits,downhill_ti,points at,uphill_count,problem
topounit,,,,,,,,,,
1,d0.g3,1,1.0,True,463.6,soil(0)+lake,-1,-1,0,


In [9]:
not_conserved = [o for o in failed if o.check == specs.WATER_CONSERVED[0] and o.signature.text() == "finite values"]
analysis.topounit_table(analysis.shrunk_domain_layout(not_conserved[-1])) if not_conserved else "no finite conservation failure"

,name,topo_number,weight,active,elevation,landunits,downhill_ti,points at,uphill_count,problem
topounit,,,,,,,,,,
1,d3.g1,1,0.0,False,158.4,soil,-1,-1,1,
2,d3.g1,2,1.0,True,361.5,soil,1,1,0,
3,d3.g2,1,0.5,True,167.9,soil,4,2,0,
4,d3.g2,2,0.5,True,112.7,soil(0)+lake,4,2,1,-> itself


Three signatures, and three different minimal domains:

* **A self-send with finite values**, shrunk to `soil | soil` (`d1.g2`): two active soil topounits,
  the lower one (102.6 m) listed second. The upper one rightly points at it; the lower one, which
  has nothing below it, points at itself. Hypothesis: *the downhill neighbour follows the order the
  topounits are listed in*: a lowest topounit not listed first still gets a neighbour.
* **A NaN flux and store**, shrunk to one topounit, `soil(0)+lake` (`d0.g3`) or `soil(0)+wetland`:
  no downhill neighbour, nothing uphill, no water ever routed, yet the flux from uphill is NaN from
  step 1. Its only soil landunit has weight 0 (ELM keeps such a landunit active). Hypothesis: *a
  topounit whose hydrology columns all weigh 0 releases its store as 0/0*. The conservation failures
  of domains 1 and 2 carry the same NaN (in the stores' sum) and shrink to the same domains.
* **Water not conserved, with finite values**, in domain 4. There all four checks fail, each its own
  way, and a smaller domain is kept only where all of them still do, so it stays at two gridcells;
  but next to the two previous patterns it shows a third: `d3.g1`'s active topounit (361.5 m) sends
  to its neighbour of weight 0 (topounit 1, inactive). Hypothesis: *water routed to an inactive
  (weight-0) topounit leaves the gridcell's water budget*.

## 3. Confirming each hypothesis

### 3.1 The downhill neighbour follows the listing order

`specs.downhill_follows_listing_order` declares the same hillslope twice, listed lowest first and
lowest last, plus a 4-topounit loop, a real gridcell of the 0.5° surface dataset listed highest first
(the Caspian coast) and a one-topounit gridcell, over a day (48 half-hour steps) of storm, steady
rain and recession. If the hypothesis holds, only the hillslopes not listed lowest first break the
routing checks:

In [10]:
order = specs.downhill_follows_listing_order(HERE / "downhill_follows_listing_order.proptest.toml")
order.to_toml(order.path)
order_parsed = order.validate()
s = order.session()
print("built domain vs declaration:", s.built().check() or "no differences")
analysis.topounit_table(s.layout)

built domain vs declaration: no differences


,name,topo_number,weight,active,elevation,landunits,downhill_ti,points at,uphill_count,problem
topounit,,,,,,,,,,
1,lowest first,1,0.30,True,50.0,soil,-1,-1,1,
2,lowest first,2,0.30,True,150.0,soil,1,1,1,
3,lowest first,3,0.40,True,300.0,soil,2,2,0,
4,lowest last,1,0.40,True,300.0,soil,5,2,0,
5,lowest last,2,0.30,True,150.0,soil,6,3,1,
6,lowest last,3,0.30,True,50.0,soil,6,3,1,-> itself
7,cycle,1,0.25,True,300.0,soil,8,2,0,
8,cycle,2,0.25,True,200.0,soil,10,4,1,
9,cycle,3,0.25,True,50.0,soil,10,4,1,-> 4 (uphill)


In [11]:
t0 = time.time()
order_outcomes = runner.run(order_parsed, case.inv, settings={"seed": 0, "max_examples": 6}, session=s,
                            out=io.StringIO())
print(f"{time.time() - t0:.0f} s")
analysis.outcome_table(order_outcomes)

115 s


where,recession,steady,storm
check,,,
IM2 never gets water to the river sooner,,pass,pass
a gridcell of one topounit is the same with IM2 off,,pass,pass
a topounit releases half its store per step,pass,pass,pass
a topounit that receives water holds at least as much,,pass,pass
no topounit sends water to itself,pass,FAIL,FAIL
the outlet passes on at least what it does without IM2,,FAIL,FAIL
the soil of the topmost topounit is the same with IM2 off,,pass,pass
the store stays finite and non-negative,pass,pass,pass
the stores gain what is sent and lose what is released,pass,pass,pass


In [12]:
storm = s.sample("storm", seed=1)
outlet_checks = [c for c in order_parsed.checks
                 if c.name in (specs.NO_SELF_SEND[0], "the outlet passes on at least what it does without IM2")]
analysis.first_failures(s, "storm", outlet_checks, storm)

,first_failing_step,failing,where
check,,,
no topounit sends water to itself,1,4,"caspian 184059 t4 (set 0), caspian 184059 t4 (set 1), lowest last t3 (set 0), lowest last t3 (set 1)"
the outlet passes on at least what it does without IM2,1,6,"caspian 184059 t4 (set 0), caspian 184059 t4 (set 1), cycle t3 (set 0), cycle t3 (set 1), lowest last t3 (set 0), lowest last t3 (set 1)"


**What it does to the runoff.** One storm, IM2 on and, in a sibling session, off; the store of each
topounit of the two listings every 4 steps (kg/m² of topounit), then the day's runoff to the river:

In [13]:
off = s.sibling({"use_im2_hillslope_hydrology": False})
on_run, off_run = s.execute("storm", storm), off.execute("storm", storm)
analysis.store_series(on_run, s.layout, ["lowest first", "lowest last"])

,lowest first t1,lowest first t2,lowest first t3,lowest last t1,lowest last t2,lowest last t3
step,,,,,,
4,1.9153e-01,4.1630e-02,0.0,0.0,2.3149e-01,2.4112e-01
8,2.0946e-01,4.4232e-02,0.0,0.0,2.3270e-01,2.5619e-01
12,2.2176e-01,4.4395e-02,0.0,0.0,2.3221e-01,2.5713e-01
16,1.3860e-02,2.7747e-03,0.0,0.0,1.4513e-02,1.6070e-02
20,8.6626e-04,1.7342e-04,0.0,0.0,9.0705e-04,1.0044e-03
24,5.4141e-05,1.0839e-05,0.0,0.0,5.6691e-05,6.2775e-05
28,3.3838e-06,6.7741e-07,0.0,0.0,3.5432e-06,3.9234e-06
32,2.1149e-07,4.2338e-08,0.0,0.0,2.2145e-07,2.4522e-07
36,1.3218e-08,2.6461e-09,0.0,0.0,1.3841e-08,1.5326e-08


In [14]:
analysis.river_table(on_run, off_run, s.layout, DTIME)

,IM2 on,IM2 off,on / off
hillslope,,,
lowest first,75.6512,76.0859,0.9943
lowest last,65.7338,66.6464,0.9863
cycle,88.2999,92.1463,0.9583
caspian 184059,5.3199,5.8209,0.9139
flat,0.1920,0.1920,1.0000


The two listings got different inputs above. To compare the topologies alone, each of 20 storms is
given to "lowest last" as "lowest first"'s inputs, element by element of the same elevation
(`analysis.mirror_example`): with IM2 off the twins must then agree to rounding.

In [15]:
storms = analysis.mirrored_storms(s, off, seeds=range(20))
print("IM2 off, the twins agree:", analysis.twins_agree_without_im2(storms))
analysis.storm_summary(storms)

IM2 off, the twins agree: True


hillslope      lowest first  lowest last
off      mean       59.8690   5.9869e+01
         min         0.5247   5.2474e-01
         max       267.6823   2.6768e+02
on       mean       58.5785   5.7803e+01
         min         0.1375   2.6363e-06
         max       266.1270   2.6549e+02
on / off mean        0.7761   6.2652e-01
         min         0.1746   3.1545e-06
         max         0.9980   9.9725e-01
half_off mean        8.6250   8.6250e+00
         min         1.0000   1.0000e+00
         max        40.0000   4.0000e+01
half_on  mean        9.7250   1.7450e+01
         min         1.0000   1.0000e+00
         max        41.0000   4.4000e+01

In [16]:
s.close()

**Confirmed.** Of the eleven checks only two fail, in the storm and the steady rain, at step 1,
and only on the hillslopes not listed lowest first: no topounit may send water to itself (the outlets
of "lowest last" and of the Caspian gridcell, which point at themselves), and the outlet must pass on
at least what it does without IM2 (those outlets, and the loop's lowest topounit, which points
uphill). "Lowest first" and "flat" never fail; every store invariant and every IM2-on/off identity
holds, which is why nothing in ELM complains. In the recession nothing new is sent, and nothing fails.

What it does: in "lowest last" the top topounit (t1) receives nothing, and the outlet (t3) stores
its own runoff along with t2's. Over 20 mirrored storms (40 twin pairs; with IM2 off the twins agree,
so the difference is the topology alone) 0.78 of the IM2-off runoff reaches the river by the end of
the day when the hillslope is listed lowest first (at least 0.17), and 0.63 when it is listed lowest
last (as little as 3e-6); half of the day's runoff has arrived by step 9.7 against 17.5 (8.6 without
IM2). The totals include subsurface drainage, which IM2 does not route, so the effect on surface
runoff is larger.

### 3.2 Water sent to an inactive topounit leaves the water budget

`specs.water_sent_to_inactive_topounit`: hillslopes listed lowest first (so the listing order plays
no part) with the outlet, the middle or the top topounit of weight 0, next to an intact control; a
24-step storm, and the same storm with floating-point exceptions trapped. If the hypothesis holds,
the checks fail where an active topounit's neighbour has weight 0 (`downhill_inactive(t)`), and
nowhere else:

In [17]:
inactive = specs.water_sent_to_inactive_topounit(HERE / "water_sent_to_inactive_topounit.proptest.toml")
inactive.to_toml(inactive.path)
inactive_outcomes = runner.run(inactive.validate(), case.inv, settings={"seed": 0}, out=io.StringIO())
analysis.outcome_table(inactive_outcomes)

where,storm,storm trapped
check,,
no active topounit sends water to an inactive one,FAIL,
the routines finish every step,,FAIL
the store stays finite and non-negative,FAIL,
the stores gain what is sent and lose what is released,FAIL,
water is conserved: soil + stores + runoff so far are the same with IM2 off,FAIL,


In [18]:
print(analysis.failure_summaries(inactive_outcomes, lines=6))

no active topounit sends water to an inactive one: implies(col_wf%qflx_to_downhill(c) > 0.0, .not. downhill_inactive(t))
  after step 1 of 24
  fails at 4 of 18 elements; first: element 5 (set 0, dead outlet, topounit 2/3, soil, column soil)
    implies = False
    col_wf%qflx_to_downhill = 8.981378454423398e-05
    downhill_inactive = True

the store stays finite and non-negative: finite(top_ws%from_uphill(t)) .and. top_ws%from_uphill(t) >= 0.0
  after step 1 of 24
  fails at 4 of 24 elements; first: element 4 (set 0, dead outlet, topounit 1/3)
    finite = False
    top_ws%from_uphill = inf
  drawn inputs at that element:

the stores gain what is sent and lose what is released: close(tend(child_wsum(top_ws%from_uphill, 'g')), dtime_mod * (child_wsum(col_wf%qflx_to_downhill, 'g') - child_wsum(col_wf%qflx_from_uphill, 'g')), rtol=1.0e-9, atol=1.0e-12)
  after step 1 of 24
  fails at 4 of 8 elements; first: element 2 (set 0, dead outlet)
    close = False
    tend = 0.04758497258577655


**Confirmed.** Every check fails at step 1, only in "dead outlet" and "dead middle" (both sets): the
active topounits whose downhill neighbour has weight 0 send to it, its store becomes `inf` (the send
divides by its weight), and the stores' balance and water conservation miss what was sent. "Dead top"
(whose weight-0 topounit sends nothing) and the control pass. With floating-point exceptions trapped,
the routines stop in step 1 (exit code -8: SIGFPE, divide by zero). On a dry step the same division
gives `0 * inf = NaN`, and ColWaterBalanceCheck's `MAX(0, NaN)` is then 0 with gfortran and NaN with
ifx (README).

### 3.3 A topounit whose hydrology columns all weigh 0 releases 0/0

`specs.no_weighted_hydrology_column`: a lake topounit as ELM builds it (the lake, and the virtual
weight-0 natural-vegetation landunit ELM keeps active on it, whose column is the topounit's only
hydrology column) at the bottom of a hillslope and alone, next to an all-soil control; a 12-step
storm. If the hypothesis holds, the flux and the store go NaN at the lake topounits only:

In [19]:
zero_weight = specs.no_weighted_hydrology_column(HERE / "no_weighted_hydrology_column.proptest.toml")
zero_weight.to_toml(zero_weight.path)
zero_weight_outcomes = runner.run(zero_weight.validate(), case.inv, settings={"seed": 0}, out=io.StringIO())
analysis.outcome_table(zero_weight_outcomes)

where,storm
check,
a topounit with water in store releases some of it,pass
the flux from uphill is finite on every hydrology column,FAIL
the store stays finite and non-negative,FAIL
water is conserved: soil + stores + runoff so far are the same with IM2 off,FAIL


In [20]:
print(analysis.failure_summaries(zero_weight_outcomes, lines=6))

the flux from uphill is finite on every hydrology column: finite(col_wf%qflx_from_uphill(c))
  after step 1 of 12
  fails at 4 of 14 elements; first: element 4 (set 0, lake low, topounit 1/3, soil, column soil)
    finite = False
    col_wf%qflx_from_uphill = nan
  drawn inputs at that element:

the store stays finite and non-negative: finite(top_ws%from_uphill(t)) .and. top_ws%from_uphill(t) >= 0.0
  after step 1 of 12
  fails at 4 of 14 elements; first: element 4 (set 0, lake low, topounit 1/3)
    finite = False
    top_ws%from_uphill = nan
  drawn inputs at that element:

water is conserved: soil + stores + runoff so far are the same with IM2 off: close(child_wsum(base%col_ws%endwb, 'g', filter%hydrologyc) + child_wsum(base%top_ws%from_uphill, 'g') + dtime_mod * acc(child_wsum(base%col_wf%qflx_runoff, 'g', filter%hydrologyc)), child_wsum(follow%col_ws%endwb, 'g', filter%hydrologyc) + dtime_mod * acc(child_wsum(follow%col_wf%qflx_runoff, 'g', filter%hydrologyc)), rtol=1.0e-9, atol=1

**Confirmed.** At step 1 the flux from uphill is NaN on the virtual soil columns of the lake
topounits ("lake low" and "lake alone", both sets), and with it their store and the conservation sum;
the all-soil control passes. "A topounit with water in store releases some of it" passes because it
cannot see this bug: a NaN store is not `> 0`, so the check finds no water in store.

## 4. Tracing them to ELM's code

**The listing order.** `set_topounit` (initGridCellsMod) looks for each topounit's highest lower
neighbour, but sets `dn_index = -1` once per gridcell, *before* the loop over its topounits: a
topounit with no lower neighbour keeps the previous topounit's answer.

In [21]:
srcroot = Path(case.inv.unit_test.e3sm_srcroot)
print(analysis.set_topounit_source(srcroot))

  315        ! find the closest downhill neighbor for each topounit
  316        dn_index = -1  ! value of -1 indicates no downhill neighbor
  317        do t1 = begt, endt
  318           t1_elev = top_pp%elevation(t1)
  319           dn_elev = min_elev
  320           do t2 = begt, endt
  321              t2_elev = top_pp%elevation(t2)
  322              if ((t2_elev < t1_elev) .and. (t2_elev >= dn_elev)) then
  323                 dn_elev = t2_elev
  324                 dn_index = t2
  325              endif
  326           end do
  327           top_pp%downhill_ti(t1)=dn_index


The synthetic domain reproduces that loop (`Layout.downhill(rule="elm")`, what the case gets as
`top_pp%downhill_ti`) and also has the intended one (`rule="intended"`: reset for every topounit).
On the declared hillslopes:

In [22]:
analysis.downhill_table(order.domain.layout())

,elevations,elm,intended,problem
hillslope,,,,
lowest first,"[50.0, 150.0, 300.0]","[-1, 1, 2]","[-1, 1, 2]",ok
lowest last,"[300.0, 150.0, 50.0]","[2, 3, 3]","[2, 3, -1]",3 -> itself
cycle,"[300.0, 200.0, 50.0, 150.0]","[2, 4, 4, 3]","[2, 4, -1, 3]",3 -> 4 (uphill)
caspian 184059,"[1.8, -20.7, -64.9, -110.7]","[2, 3, 4, 4]","[2, 3, 4, -1]",4 -> itself
flat,[100.0],[-1],[-1],ok


ELM itself does the same in a real run: the reference of `hs_season` captured `downhill_ti` in the
7 gridcells of the 0.5° dataset that list their topounits highest first (Salton Sea and Caspian
coast); every outlet points at itself:

In [23]:
analysis.captured_downhill(case)

,gindex,lat,lon,topounit,number in gridcell,weight,downhill_ti,points at itself
0,177249,33.25,-115.75,1,1,0.9002,2,False
1,177249,33.25,-115.75,2,2,0.0499,3,False
2,177249,33.25,-115.75,3,3,0.0500,3,True
3,184059,37.75,49.25,4,1,0.7007,5,False
4,184059,37.75,49.25,5,2,0.1497,6,False
5,184059,37.75,49.25,6,3,0.0996,7,False
6,184059,37.75,49.25,7,4,0.0500,7,True
7,186937,39.75,48.25,8,1,0.9506,9,False
8,186937,39.75,48.25,9,2,0.0494,9,True
9,187666,40.25,52.75,10,1,0.9501,11,False


**The inactive target.** The search runs over every topounit, weight-0 ones included, and
`HydrologyDrainage` scales the sent water by the target's weight in the gridcell, 0 for an inactive
topounit:

In [24]:
print(analysis.send_to_downhill_source(srcroot))

  312                 qflx_h2osfc_surf(c) = qflx_h2osfc_surf(c) - temp_to_downhill
  313  
  314                 ! update the downhill topounit water state
  315                 ! relative weight of this column to downhill topounit on the gridcell is used to 
  316                 ! scale the mass of water from this column to a potentially larger or smaller 
  317                 ! downhill topounit
  318                 temp_mass = (qflx_to_downhill(c) * dtime) * (col_pp%wtgcell(c) / top_pp%wtgcell(downhill_t))
  319                 top_ws%from_uphill(downhill_t) = top_ws%from_uphill(downhill_t) + temp_mass
  320              else


**The 0/0 release.** `SurfaceRunoff` gives each hydrology column of a topounit its weight over the
summed weight of the topounit's hydrology columns. ELM keeps a weight-0 soil landunit active on every
topounit that is not all glacier (subgridWeightsMod is_active_l); on a lake or wetland topounit its
column is the only hydrology column, so `uphill_wt(t)` is 0 and the release is 0/0 every step:

In [25]:
print(analysis.release_from_store_source(srcroot))

  268              c = filter_hydrologyc(fc)
  269              t = col_pp%topounit(c)
  270              top_pp%uphill_wt(t) = 0._r8
  271           end do
  272           ! Next sum the relevant column weights
  273           do fc = 1, num_hydrologyc
  274              c = filter_hydrologyc(fc)
  275              t = col_pp%topounit(c)
  276              top_pp%uphill_wt(t) = top_pp%uphill_wt(t) + col_pp%wttopounit(c)
  277           end do
  278  
  279           ! flow from uphill topounit goes to top of soil for soil, crop, and pervious road columns
  280           do fc = 1, num_hydrologyc
  281              c = filter_hydrologyc(fc)
  282              t = col_pp%topounit(c)
  283              qflx_from_uphill(c) = (col_pp%wttopounit(c)/top_pp%uphill_wt(t)) * (frac_from_uphill * top_ws%from_uphill(t)) / dtime
  284              qflx_top_soil(c) = qflx_top_soil(c) + qflx_from_uphill(c)


## 5. Proving the diagnosis

If the self-sends come from set_topounit's loop and nothing else, the same property test on a domain
that gives the routines the intended neighbours (`[domain] downhill = "intended"`, what an ELM with
the loop fixed builds) must lose exactly those failures, and keep the other two bugs'. The same
seed draws the same four domains:

In [26]:
fixed = specs.random_hillslopes(HERE / "random_hillslopes.proptest.toml", downhill="intended")
t0 = time.time()
fixed_outcomes = runner.run(fixed.validate(), case.inv, settings={"seed": 0}, out=io.StringIO())
print(f"{time.time() - t0:.0f} s")
analysis.compare_outcomes(elm=outcomes, intended=fixed_outcomes)

225 s


elm                            intended                           
where                                                                       domain 1 domain 2 domain 3 domain 4 domain 1 domain 2 domain 3 domain 4
check                                                                                                                                              
no topounit sends water to itself                                               pass     FAIL     pass     FAIL     pass     pass     pass     pass
the flux from uphill is finite on every hydrology column                        FAIL     FAIL     pass     FAIL     FAIL     FAIL     pass     FAIL
the store stays finite and non-negative                                         FAIL     FAIL     pass     FAIL     FAIL     FAIL     pass     FAIL
water is conserved: soil + stores + runoff so far are the same with IM2 off     FAIL     FAIL     pass     FAIL     FAIL     FAIL     pass     FAIL

In [27]:
analysis.shrunk_domains(fixed_outcomes)

signature                                     shrunk_to
domain check                                                                                                                                                                       
1      the flux from uphill is finite on every hydrology column                                   non-finite: col_wf%qflx_from_uphill                           d0.g3: soil(0)+lake
       the store stays finite and non-negative                                                         non-finite: top_ws%from_uphill                           d0.g3: soil(0)+lake
       water is conserved: soil + stores + runoff so far are the same with IM2 off  non-finite: child_wsum(base%top_ws%from_uphill,g)                           d0.g3: soil(0)+lake
2      the flux from uphill is finite on every hydrology column                                   non-finite: col_wf%qflx_from_uphill                        d1.g2: soil(0)+wetland
       the store stays finite and non-negative                                                         non-finite: top_ws%from_uphill                        d1.g2: soil(0)+wetland
       water is conserved: soil + stores + runoff so far are the same with IM2 off  non-finite: child_wsum(base%top_ws%from_uphill,g)                        d1.g2: soil(0)+wetland
4      the flux from uphill is finite on every hydrology column                                   non-finite: col_wf%qflx_from_uphill  d3.g1: soil [w0] | soil; d3.g3: soil(0)+lake
       the store stays finite and non-negative                                                         non-finite: top_ws%from_uphill  d3.g1: soil [w0] | soil; d3.g3: soil(0)+lake
       water is conserved: soil + stores + runoff so far are the same with IM2 off                                      finite values  d3.g1: soil [w0] | soil; d3.g3: soil(0)+lake

Exactly the self-send failures are gone (domains 2 and 4). Every other failure remains, on the same
domains, shrunk to the same small domains with the same signatures. So the listing order explains the
self-sends and nothing else, and the other two hypotheses are independent bugs: they are what is left
to fix after set_topounit.

## 6. Summary

A property test written without knowing any bug (four invariants of IM2 routing on random
hillslopes, a 12-step storm) found three bugs in about 5 minutes on a laptop, and the signatures of its
shrunk failures separated them:

| bug | found as | confirmed by | in ELM's code | fix direction |
|---|---|---|---|---|
| the downhill neighbour follows the listing order | a self-send in `soil \| soil` | `downhill_follows_listing_order`: only hillslopes not listed lowest first fail; 0.63 vs 0.78 of the runoff reaches the river, half of it 8 steps later | `set_topounit` resets `dn_index` once per gridcell | reset it for every topounit (one line) |
| water sent to an inactive topounit leaves the budget | conservation fails with finite values next to a weight-0 topounit | `water_sent_to_inactive_topounit`: only active topounits with a weight-0 neighbour; `inf` store; SIGFPE when trapped | the search ignores activity; the send divides by the target's weight | skip inactive topounits in the search, or guard the send |
| a topounit whose hydrology columns weigh 0 releases 0/0 | NaN flux in `soil(0)+lake` | `no_weighted_hydrology_column`: only the lake topounits | `SurfaceRunoff` divides by `uphill_wt(t)` = 0 | guard the release with `uphill_wt(t) > 0`, and decide where such a topounit's water goes |

`[domain] downhill = "intended"` proves the first diagnosis: the self-sends disappear and the other two
bugs remain. `README.md` has what each bug does in real ELM data and runs (the 7 self-looping gridcells
of the 0.5° dataset, the 1527 NaN stores of a 5-day run).